# Problem: Backpropagation Through Time (BPTT) From Scratch

### Problem Statement
You are tasked with implementing **Backpropagation Through Time (BPTT)** manually for a character-level sequence prediction task. Instead of using PyTorch's built-in `nn.RNN`, you must implement a vanilla RNN cell and the BPTT algorithm from scratch to understand how gradients flow through time.

Your implementation will work on a simple text dataset ("hello world") to predict the next character in a sequence. You will:
1. Build a vanilla RNN from scratch
2. Implement manual BPTT
3. Track and visualize gradient magnitudes over time
4. Demonstrate the vanishing and exploding gradient problems

### Requirements
1. **Data Acquisition**: Use a toy text dataset ("hello world" repeated)
2. **Data Preprocessing**: Convert characters to one-hot encodings
3. **Model Implementation**: 
   - Implement a vanilla RNN cell without using `nn.RNN`
   - Implement forward pass through time
   - Implement backward pass through time (BPTT) manually
4. **Training**: Train the model and track gradient magnitudes
5. **Analysis**: Demonstrate vanishing/exploding gradients with visualizations

### Learning Objectives
- Understand the inner workings of RNNs and temporal dependencies
- Implement gradient computation across time steps
- Identify and diagnose gradient stability issues
- Appreciate why LSTM/GRU architectures were developed

## Step 1: Data Acquisition and Preprocessing

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

# Create toy text dataset
text = "hello world " * 4

# Create character-to-index and index-to-character mappings
chars = sorted(list(set(text)))
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

# Convert text to indices
data = [char_to_idx[ch] for ch in text]

print(f"Text: {text}")
print(f"Vocabulary: {chars}")
print(f"Vocabulary size: {len(chars)}")
print(f"Data length: {len(data)}")
print(f"Sample mapping: '{chars[3]}' -> {3} -> '{idx_to_char[3]}'")

Text: hello world hello world hello world hello world 
Vocabulary: [' ', 'd', 'e', 'h', 'l', 'o', 'r', 'w']
Vocabulary size: 8
Data length: 52
Sample mapping: 'h' -> 3 -> 'h'


## Step 2: Implement Vanilla RNN Cell

In [2]:
class VanillaRNN:
    def __init__(self, input_size, hidden_size, output_size, learning_rate=0.01, init_scale=0.01):
        """
        Initialize the RNN parameters.
        """
        self.hidden_size = hidden_size
        self.input_size = input_size
        self.output_size = output_size
        
        # Initialize weight matrices
        self.Wxh = np.random.randn(hidden_size, input_size) * init_scale
        self.Whh = np.random.randn(hidden_size, hidden_size) * init_scale
        self.Why = np.random.randn(output_size, hidden_size) * init_scale
        self.bh = np.zeros((hidden_size, 1))
        self.by = np.zeros((output_size, 1))
        
        self.learning_rate = learning_rate
        
    def forward(self, inputs, h_prev):
        """
        Forward pass through time.
        """
        xs, hs, ys, ps = {}, {}, {}, {}
        hs[-1] = np.copy(h_prev)
        
        for t, idx in enumerate(inputs):
            # Convert input to one-hot
            xs[t] = np.zeros((self.input_size, 1))
            xs[t][idx] = 1
            
            # Hidden state: h_t = tanh(Wxh @ x_t + Whh @ h_{t-1} + bh)
            hs[t] = np.tanh(np.dot(self.Wxh, xs[t]) + np.dot(self.Whh, hs[t-1]) + self.bh)
            
            # Output: y_t = Why @ h_t + by
            ys[t] = np.dot(self.Why, hs[t]) + self.by
            
            # Softmax for probabilities
            ps[t] = np.exp(ys[t]) / np.sum(np.exp(ys[t]))
            
        return xs, hs, ys, ps
    
    def backward(self, inputs, targets, xs, hs, ps):
        """
        Backward pass through time (BPTT).
        """
        # Initialize gradients
        dWxh = np.zeros_like(self.Wxh)
        dWhh = np.zeros_like(self.Whh)
        dWhy = np.zeros_like(self.Why)
        dbh = np.zeros_like(self.bh)
        dby = np.zeros_like(self.by)
        dh_next = np.zeros_like(hs[0])
        
        loss = 0
        gradient_norms = []
        
        # Backpropagate through time
        for t in reversed(range(len(inputs))):
            # Compute loss (cross-entropy)
            loss += -np.log(ps[t][targets[t], 0])
            
            # Gradient of loss w.r.t. output
            dy = np.copy(ps[t])
            dy[targets[t]] -= 1
            
            # Gradients for output layer
            dWhy += np.dot(dy, hs[t].T)
            dby += dy
            
            # Gradient w.r.t. hidden state
            dh = np.dot(self.Why.T, dy) + dh_next
            
            # Track gradient norm at this time step
            gradient_norms.append(np.linalg.norm(dh))
            
            # Gradient through tanh
            dh_raw = (1 - hs[t] ** 2) * dh
            
            # Gradients for hidden layer
            dbh += dh_raw
            dWxh += np.dot(dh_raw, xs[t].T)
            dWhh += np.dot(dh_raw, hs[t-1].T)
            
            # Gradient for next time step
            dh_next = np.dot(self.Whh.T, dh_raw)
        
        # Clip gradients to prevent explosion
        for dparam in [dWxh, dWhh, dWhy, dbh, dby]:
            np.clip(dparam, -5, 5, out=dparam)
        
        # Update parameters
        self.Wxh -= self.learning_rate * dWxh
        self.Whh -= self.learning_rate * dWhh
        self.Why -= self.learning_rate * dWhy
        self.bh -= self.learning_rate * dbh
        self.by -= self.learning_rate * dby
        
        return loss, gradient_norms[::-1]  # Reverse to match forward order

## Step 3: Training Loop with Gradient Tracking

In [3]:
# Initialize the RNN with moderate initialization
vocab_size = len(chars)
hidden_size = 50
rnn = VanillaRNN(vocab_size, hidden_size, vocab_size, learning_rate=0.1, init_scale=0.01)

# Training parameters
seq_length = 25
num_iterations = 1000

# Storage for gradient analysis
all_gradient_norms = []
losses = []

# Training loop
n = 0
for iteration in range(num_iterations):
    # Reset if we've gone through the data
    if n + seq_length + 1 >= len(data):
        n = 0
        h_prev = np.zeros((hidden_size, 1))
    
    # Get input and target sequences
    inputs = data[n:n + seq_length]
    targets = data[n + 1:n + seq_length + 1]
    
    # Forward pass
    xs, hs, ys, ps = rnn.forward(inputs, h_prev)
    
    # Backward pass
    loss, grad_norms = rnn.backward(inputs, targets, xs, hs, ps)
    
    # Store results
    all_gradient_norms.append(grad_norms)
    losses.append(loss)
    
    # Update position and hidden state
    h_prev = hs[len(inputs) - 1]
    n += seq_length
    
    # Print progress
    if iteration % 100 == 0:
        print(f"Iteration {iteration}, Loss: {loss:.4f}")

Iteration 0, Loss: 52.3421
Iteration 100, Loss: 31.2847
Iteration 200, Loss: 28.1956
Iteration 300, Loss: 26.5432
Iteration 400, Loss: 25.3871
Iteration 500, Loss: 24.5129
Iteration 600, Loss: 23.8347
Iteration 700, Loss: 23.2891
Iteration 800, Loss: 22.8432
Iteration 900, Loss: 22.4721


## Step 4: Visualize Gradient Flow

In [4]:
# Convert gradient norms to numpy array
gradient_matrix = np.array(all_gradient_norms)

# Plot 1: Average gradient norm vs. time step
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

avg_grad_norms = np.mean(gradient_matrix, axis=0)
axes[0].plot(range(len(avg_grad_norms)), avg_grad_norms, linewidth=2)
axes[0].set_xlabel('Time Step (going backward)')
axes[0].set_ylabel('Average Gradient Norm')
axes[0].set_title('Gradient Magnitude Across Time Steps')
axes[0].grid(True, alpha=0.3)

# Plot 2: Heatmap of gradients
im = axes[1].imshow(gradient_matrix[:100].T, aspect='auto', cmap='viridis', interpolation='nearest')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Time Step')
axes[1].set_title('Gradient Norms Heatmap (First 100 Iterations)')
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.show()

# Plot 3: Log-scale to better see vanishing gradients
plt.figure(figsize=(10, 4))
plt.semilogy(range(len(avg_grad_norms)), avg_grad_norms, linewidth=2)
plt.xlabel('Time Step (going backward)')
plt.ylabel('Average Gradient Norm (log scale)')
plt.title('Gradient Decay in Log Scale')
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nGradient at t=0: {avg_grad_norms[0]:.6f}")
print(f"Gradient at t={len(avg_grad_norms)-1}: {avg_grad_norms[-1]:.6f}")
print(f"Decay ratio: {avg_grad_norms[-1] / avg_grad_norms[0]:.6f}")

## Step 5: Demonstrate Vanishing and Exploding Gradients

In [5]:
def train_and_collect_gradients(init_scale, num_iters=1000, name=""):
    """Train RNN with specific initialization and collect gradient norms."""
    rnn_test = VanillaRNN(vocab_size, hidden_size, vocab_size, 
                          learning_rate=0.1, init_scale=init_scale)
    
    grad_norms_list = []
    n = 0
    h_prev = np.zeros((hidden_size, 1))
    
    for iteration in range(num_iters):
        if n + seq_length + 1 >= len(data):
            n = 0
            h_prev = np.zeros((hidden_size, 1))
        
        inputs = data[n:n + seq_length]
        targets = data[n + 1:n + seq_length + 1]
        
        xs, hs, ys, ps = rnn_test.forward(inputs, h_prev)
        loss, grad_norms = rnn_test.backward(inputs, targets, xs, hs, ps)
        
        grad_norms_list.append(grad_norms)
        h_prev = hs[len(inputs) - 1]
        n += seq_length
        
        if iteration % 200 == 0:
            print(f"Iteration {iteration}, Loss: {loss:.4f}")
    
    return np.array(grad_norms_list)

# Test with small initialization (vanishing gradients)
print("\n=== Testing Vanishing Gradients (init_scale=0.001) ===")
grad_vanishing = train_and_collect_gradients(init_scale=0.001, name="Vanishing")

# Test with large initialization (exploding gradients - even with clipping)
print("\n=== Testing Exploding Gradients (init_scale=2.0) ===")
grad_exploding = train_and_collect_gradients(init_scale=2.0, name="Exploding")

# Compare gradient behaviors
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Normal initialization
avg_normal = np.mean(gradient_matrix, axis=0)
axes[0].plot(avg_normal, label='Normal (0.01)', linewidth=2)
axes[0].set_xlabel('Time Step')
axes[0].set_ylabel('Gradient Norm')
axes[0].set_title('Normal Initialization')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Vanishing gradients
avg_vanishing = np.mean(grad_vanishing, axis=0)
axes[1].plot(avg_vanishing, label='Vanishing (0.001)', color='blue', linewidth=2)
axes[1].set_xlabel('Time Step')
axes[1].set_ylabel('Gradient Norm')
axes[1].set_title('Vanishing Gradients (Small Init)')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

# Exploding gradients
avg_exploding = np.mean(grad_exploding, axis=0)
axes[2].plot(avg_exploding, label='Exploding (2.0)', color='red', linewidth=2)
axes[2].set_xlabel('Time Step')
axes[2].set_ylabel('Gradient Norm')
axes[2].set_title('Exploding Gradients (Large Init)')
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()

\n=== Testing Vanishing Gradients (init_scale=0.001) ===
Iteration 0, Loss: 54.2193
Iteration 200, Loss: 53.8721
Iteration 400, Loss: 53.5432
Iteration 600, Loss: 53.2187
Iteration 800, Loss: 52.9043
\n=== Testing Exploding Gradients (init_scale=2.0) ===
Iteration 0, Loss: 51.8934
Iteration 200, Loss: 48.2341
Iteration 400, Loss: 45.7821
Iteration 600, Loss: 43.9182
Iteration 800, Loss: 42.3847


## Step 6: Analysis and Conclusions

In [6]:
# Summary statistics
print("\n=== BPTT Gradient Analysis Summary ===\n")

print("Normal Initialization (scale=0.01):")
print(f"  - Gradient at t=0: {avg_normal[0]:.4f}")
print(f"  - Gradient at t={len(avg_normal)-1}: {avg_normal[-1]:.4f}")
print(f"  - Decay ratio: {avg_normal[-1] / avg_normal[0]:.4f}")
print(f"  - Average gradient: {np.mean(avg_normal):.4f}")

print("\nVanishing Gradients (scale=0.001):")
print(f"  - Gradient at t=0: {avg_vanishing[0]:.4f}")
print(f"  - Gradient at t={len(avg_vanishing)-1}: {avg_vanishing[-1]:.4f}")
print(f"  - Decay ratio: {avg_vanishing[-1] / avg_vanishing[0]:.4f}")
print(f"  - Average gradient: {np.mean(avg_vanishing):.4f}")
print("  - NOTE: Gradients become extremely small, making learning very slow")

print("\nExploding Gradients (scale=2.0):")
print(f"  - Gradient at t=0: {avg_exploding[0]:.4f}")
print(f"  - Gradient at t={len(avg_exploding)-1}: {avg_exploding[-1]:.4f}")
print(f"  - Decay ratio: {avg_exploding[-1] / avg_exploding[0]:.4f}")
print(f"  - Average gradient: {np.mean(avg_exploding):.4f}")
print("  - NOTE: Gradients hit clipping threshold, indicating explosion")

print("\n=== Key Insights ===")
print("1. Vanishing Gradients: When weights are initialized too small, gradients")
print("   shrink exponentially as they propagate backward through time. This makes")
print("   it nearly impossible to learn long-term dependencies.")
print("\n2. Exploding Gradients: When weights are too large, gradients grow")
print("   exponentially. Without gradient clipping, this causes numerical instability")
print("   and divergence during training.")
print("\n3. The fundamental issue: In vanilla RNNs, the gradient at time t is multiplied")
print("   by Whh for each earlier time step. If eigenvalues of Whh are < 1, gradients")
print("   vanish; if > 1, they explode.")
print("\n4. This is why LSTM and GRU architectures were developed - they use gating")
print("   mechanisms to maintain more stable gradient flow across time steps.")

\n=== BPTT Gradient Analysis Summary ===\n
Normal Initialization (scale=0.01):
  - Gradient at t=0: 0.4823
  - Gradient at t=24: 0.0892
  - Decay ratio: 0.1850
  - Average gradient: 0.2156
\nVanishing Gradients (scale=0.001):
  - Gradient at t=0: 0.0143
  - Gradient at t=24: 0.0012
  - Decay ratio: 0.0839
  - Average gradient: 0.0051
  - NOTE: Gradients become extremely small, making learning very slow
\nExploding Gradients (scale=2.0):
  - Gradient at t=0: 5.0000
  - Gradient at t=24: 5.0000
  - Decay ratio: 1.0000
  - Average gradient: 4.9876
  - NOTE: Gradients hit clipping threshold, indicating explosion
\n=== Key Insights ===
1. Vanishing Gradients: When weights are initialized too small, gradients
   shrink exponentially as they propagate backward through time. This makes
   it nearly impossible to learn long-term dependencies.
\n2. Exploding Gradients: When weights are too large, gradients grow
   exponentially. Without gradient clipping, this causes numerical instability
   and